# Aula 8 — Padrões, outliers e formulação de hipóteses a partir dos dados
**Fund. de Programação, Dados e Estatística para IA** · Einstein Hospital Israelita

Objetivo: avançar da descrição de variáveis isoladas para uma investigação estruturada das relações na base.

> EDA gera pistas e hipóteses. Não prova causalidade nem substitui inferência estatística.

## Revisão da aula anterior
Antes de iniciarmos o conteúdo de hoje, vamos recapitular com exercícios alguns conceitos fundamentais apresentados na última aula.

In [ ]:
import pandas as pd
import numpy as np

### Exercício 1 — Frequências e ausências
Compare `value_counts(normalize=True)` com `value_counts(normalize=True, dropna=False)`. Qual é o percentual de Cardio em cada caso e por que ele muda?

In [ ]:
df_rev = pd.DataFrame({
    "servico": ["Cardio", "Neuro", "Cardio", None, "Orto"]
})
# Sua resposta aqui

### Exercício 2 — Média e mediana
Calcule `mean()` e `median()`. Qual medida representa melhor o tempo típico de internação? O que a distância entre elas sugere?

In [ ]:
df_rev = pd.DataFrame({
    "tempo_internacao_dias": [2, 3, 3, 4, 28]
})
# Sua resposta aqui

### Exercício 3 — Dispersão e posição
Calcule a amplitude, o desvio-padrão com `std()` e o IQR com `quantile()`. Qual dessas medidas não muda se 95 for substituído por 120?

In [ ]:
df_rev = pd.DataFrame({
    "idade": [30, 40, 50, 60, 70, 80, 95]
})
# Sua resposta aqui

### Exercício 4 — Centro por grupo
Com `groupby()` e `agg()`, obtenha por serviço: n (`size`), `tempo_medio` e `tempo_mediano`. Por que comparar apenas as médias levaria a uma leitura enganosa?

In [ ]:
df_rev = pd.DataFrame({
    "servico": ["Cardio", "Cardio", "Cardio", "Neuro", "Neuro"],
    "tempo_internacao_dias": [3, 4, 20, 5, 6]
})
# Sua resposta aqui

### Soluções da revisão

In [ ]:
# Exercício 1
s = pd.Series(["Cardio", "Neuro", "Cardio", None, "Orto"], name="servico")
print(s.value_counts(normalize=True).mul(100))
print(s.value_counts(normalize=True, dropna=False).mul(100))
# 50% com 4 valores válidos; 40% quando a ausência entra no denominador (5)

In [ ]:
# Exercício 2
t = pd.Series([2, 3, 3, 4, 28])
print("Média:", t.mean(), "| Mediana:", t.median())
# A mediana (3) representa melhor o tempo típico; o 28 puxa a média para 8

In [ ]:
# Exercício 3
for valores in ([30, 40, 50, 60, 70, 80, 95], [30, 40, 50, 60, 70, 80, 120]):
    x = pd.Series(valores)
    iqr = x.quantile(0.75) - x.quantile(0.25)
    print("Amplitude:", x.max() - x.min(), "| DP:", round(x.std(), 2), "| IQR:", iqr)
# O IQR não muda: depende só dos 50% centrais

In [ ]:
# Exercício 4
df_rev = pd.DataFrame({
    "servico": ["Cardio", "Cardio", "Cardio", "Neuro", "Neuro"],
    "tempo_internacao_dias": [3, 4, 20, 5, 6]
})
df_rev.groupby("servico").agg(
    n=("tempo_internacao_dias", "size"),
    tempo_medio=("tempo_internacao_dias", "mean"),
    tempo_mediano=("tempo_internacao_dias", "median")
)
# Pela média Cardio parece mais longo; pela mediana, não. Um único valor e grupos pequenos sustentam a diferença

## 01 · Das variáveis às relações
**Onde a Aula 7 terminou:** um tempo de internação muito alto é apenas um valor extremo ou pertence a um perfil específico de paciente?

Um padrão tem várias explicações possíveis: diferença real entre populações, composição dos grupos, viés de seleção, qualidade dos dados, acaso e fatores não medidos. O papel da EDA é localizar a pergunta relevante.

### Leitura e verificação inicial
Toda análise começa confirmando qual arquivo foi carregado.

In [ ]:
df = pd.read_csv("pacientes_aula8_analise.csv")
df.head()

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

## 02 · Cruzando categorias
### Serviço × readmissão
**Pergunta para a turma:** mais readmissões em número absoluto significam uma taxa maior?

In [ ]:
tabela = pd.crosstab(df["servico"], df["readmissao_30d"])
tabela

### De contagens para proporções
`normalize="index"` faz cada linha somar 100%. Uma proporção alta num grupo pequeno pede cautela.

In [ ]:
tabela_pct = pd.crosstab(
    df["servico"],
    df["readmissao_30d"],
    normalize="index"
).mul(100).round(1)
tabela_pct

### Margens mantêm o denominador visível

In [ ]:
pd.crosstab(
    df["servico"],
    df["readmissao_30d"],
    margins=True
)

## 03 · Comparando grupos
### Tempo de internação por serviço
Compare média e mediana. Uma diferença entre médias não mostra que o serviço é a causa.

In [ ]:
resumo_servico = df.groupby("servico")["tempo_internacao_dias"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)
resumo_servico.round(2)

### Pacientes com e sem readmissão
A resposta é descritiva: “na amostra observada, o grupo apresenta valores maiores”. Significância fica para a Aula 13.

In [ ]:
df.groupby("readmissao_30d")[[
    "idade",
    "tempo_internacao_dias",
    "internacoes_anteriores"
]].agg(["count", "mean", "median", "std"]).round(2)

## 04 · Composição dos grupos
### Agrupando por mais de uma variável
Grupos menores revelam estrutura escondida pela média global, mas tornam as estimativas instáveis.

In [ ]:
df.groupby(["servico", "readmissao_30d"])["tempo_internacao_dias"].agg(
    ["count", "mean", "median"]
).round(2)

### Estratificar pode mudar a história
Um padrão agregado pode enfraquecer, desaparecer ou inverter. Estratifique guiado por uma pergunta plausível.

In [ ]:
pd.crosstab(
    [df["servico"], df["faixa_etaria"]],
    df["readmissao_30d"],
    normalize="index"
).mul(100).round(1)

## 05 · Valores extremos em contexto
### Comece pela ordenação
As outras colunas ajudam a julgar se o valor é plausível.

In [ ]:
df.nlargest(5, "tempo_internacao_dias")[[
    "idade",
    "servico",
    "tempo_internacao_dias",
    "internacoes_anteriores",
    "readmissao_30d"
]]

In [ ]:
df.nsmallest(5, "tempo_internacao_dias")[[
    "idade",
    "servico",
    "tempo_internacao_dias",
    "readmissao_30d"
]]

### Regra do IQR

In [ ]:
q1 = df["tempo_internacao_dias"].quantile(0.25)
q3 = df["tempo_internacao_dias"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
print("Q1:", q1, "| Q3:", q3, "| IQR:", iqr)
print("Limites:", limite_inferior, limite_superior)

### Selecionando os valores sinalizados
A regra identifica valores distantes da região central da distribuição.

In [ ]:
mask_outlier = (
    (df["tempo_internacao_dias"] < limite_inferior) |
    (df["tempo_internacao_dias"] > limite_superior)
)
outliers_tempo = df.loc[mask_outlier]
outliers_tempo

> “Outlier estatístico” não é sinônimo de “dado inválido”.

### Sinalizar sem apagar
Preserva todas as observações, permite comparar resultados com e sem extremos, mantém a regra rastreável e mostra se os extremos se concentram em algum grupo.

In [ ]:
df["outlier_tempo_iqr"] = mask_outlier
df["outlier_tempo_iqr"].value_counts()

### Quem são as observações sinalizadas
Concentram-se num serviço, numa faixa etária? Têm mais internações anteriores?

In [ ]:
df.loc[df["outlier_tempo_iqr"], [
    "idade", "faixa_etaria", "servico",
    "tempo_internacao_dias",
    "internacoes_anteriores", "readmissao_30d"
]].sort_values("tempo_internacao_dias", ascending=False)

### Comparando extremos e demais
O resultado sugere uma pergunta. Não mostra que o tempo extremo provoca readmissão.

In [ ]:
df.groupby("outlier_tempo_iqr")[[
    "idade",
    "internacoes_anteriores",
    "readmissao_30d"
]].agg(["count", "mean", "median"]).round(2)

### Os extremos mexem mais na média
O cálculo avalia sensibilidade. Não justifica exclusão.

In [ ]:
sem_extremos = df.loc[~df["outlier_tempo_iqr"]]
tempo_todos = df["tempo_internacao_dias"]
tempo_sem = sem_extremos["tempo_internacao_dias"]
print("Média:", tempo_todos.mean(), "→", tempo_sem.mean())
print("Mediana:", tempo_todos.median(), "→", tempo_sem.median())

### Outlier não é propriedade do paciente
20 dias pode ser raro na população geral e esperado num subgrupo de alta complexidade.

**Pergunta para a turma:** faz sentido aplicar o mesmo limite a todos os serviços?

In [ ]:
df.groupby("servico")["tempo_internacao_dias"].quantile(
    [0.25, 0.50, 0.75]
).unstack()

## 06 · Do padrão à hipótese
Observação → Pergunta → Hipótese preliminar → Alternativas → Evidência necessária

**Preferir:** “Na amostra observada, o serviço A apresentou maior proporção de readmissão.” “Esse padrão sugere uma hipótese a investigar.”

**Evitar:** “O serviço A causa mais readmissões.” “Os outliers são erros e devem ser removidos.”

### Registro de achados
Preencha um bloco por achado.

- **Achado:** 
- **Evidência:** 
- **Hipótese:** 
- **Limitação:** 
- **Próximo passo:**

### Antes de concluir
Verifique grupos, distribuição, extremos e composição.

**Próxima aula — Aula 9:** escolhendo a visualização certa.